In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import mixed_precision

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                            precision_recall_fscore_support)
from sklearn.neighbors import NearestNeighbors
from sklearn.utils.class_weight import compute_class_weight

import time
import gc
import os
from collections import Counter
import pickle


In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ GPU memory growth enabled")
    except RuntimeError as e:
        print(e)

# Enable XLA compilation for faster training
tf.config.optimizer.set_jit(True)
print("✅ XLA JIT compilation enabled")

# Enable mixed precision for memory efficiency
#policy = mixed_precision.Policy('mixed_float16')
#mixed_precision.set_global_policy(policy)
#print("✅ Mixed precision enabled (float16)")

In [ ]:

class FullDatasetLoader:
    """Loads ALL 10 CSV files with ALL features using memory mapping"""
    
    def __init__(self, data_path, chunk_size=100000):
        self.data_path = data_path
        self.chunk_size = chunk_size
        self.scaler = None
        self.label_encoder = None
        self.feature_names = None
        self.num_features = None
        
    def load_full_dataset(self):
        """Load ALL 10 CSV files with ALL features"""
        
        print("\n" + "="*80)
        print("LOADING FULL DATASET (ALL 10 CSV FILES)")
        print("="*80)
        
        csv_files = sorted([f for f in os.listdir(self.data_path) if f.endswith('.csv')])
        print(f"Found {len(csv_files)} CSV files")
        print(f"Files: {csv_files}")
        
        # First pass: collect metadata and determine feature names
        print("\n📊 Pass 1: Collecting metadata from all files...")
        all_columns = None
        total_rows = 0
        
        for i, file in enumerate(csv_files, 1):
            filepath = os.path.join(self.data_path, file)
            print(f"Scanning {i}/{len(csv_files)}: {file}")
            
            # Read just first chunk to get columns
            df_sample = pd.read_csv(filepath, nrows=1000)
            df_sample = df_sample[df_sample['Label'] != 'Label']
            
            if all_columns is None:
                all_columns = df_sample.columns.tolist()
            
            # Count total rows
            row_count = sum(1 for _ in open(filepath)) - 1  # -1 for header
            total_rows += row_count
            print(f"  Rows: {row_count:,}")
        
        print(f"\n✅ Total rows across all files: {total_rows:,}")
        print(f"✅ Total columns: {len(all_columns)}")
        
        # Remove non-feature columns
        cols_to_remove = ['Timestamp', 'Flow ID', 'Source IP', 'Destination IP', 
                         'Source Port', 'Destination Port', 'Protocol']
        feature_cols = [c for c in all_columns if c not in cols_to_remove and c != 'Label']
        
        self.feature_names = feature_cols
        self.num_features = len(feature_cols)
        
        print(f"✅ Feature columns (ALL): {self.num_features}")
        print(f"✅ Using ALL features - NO feature selection")
        
        # Second pass: Load all data in chunks and save to memory-mapped file
        print("\n📂 Pass 2: Loading all data and creating memory-mapped arrays...")
        
        # Create temporary memory-mapped files
        X_memmap = np.memmap('X_temp.dat', dtype='float32', mode='w+', 
                            shape=(total_rows, self.num_features))
        y_memmap = np.memmap('y_temp.dat', dtype='<U50', mode='w+', 
                            shape=(total_rows,))
        
        current_idx = 0
        
        for i, file in enumerate(csv_files, 1):
            filepath = os.path.join(self.data_path, file)
            print(f"\n{'='*60}")
            print(f"Processing {i}/{len(csv_files)}: {file}")
            print(f"{'='*60}")
            
            chunk_iter = pd.read_csv(filepath, chunksize=self.chunk_size)
            file_rows = 0
            
            for chunk_num, chunk in enumerate(chunk_iter):
                # Clean chunk
                chunk = chunk[chunk['Label'] != 'Label']
                
                if len(chunk) == 0:
                    continue
                
                # Drop non-feature columns
                chunk = chunk.drop(columns=[c for c in cols_to_remove if c in chunk.columns], 
                                  errors='ignore')
                
                # Separate features and labels
                y_chunk = chunk['Label'].values
                X_chunk = chunk[feature_cols]
                
                # Convert to numeric
                X_chunk = X_chunk.apply(pd.to_numeric, errors='coerce')
                X_chunk = X_chunk.replace([np.inf, -np.inf], np.nan)
                X_chunk = X_chunk.fillna(method='ffill')
                X_chunk = X_chunk.fillna(0)
                
                # Store in memmap
                n_samples = len(X_chunk)
                X_memmap[current_idx:current_idx + n_samples] = X_chunk.values.astype(np.float32)
                y_memmap[current_idx:current_idx + n_samples] = y_chunk
                
                current_idx += n_samples
                file_rows += n_samples
                
                if chunk_num % 10 == 0:
                    print(f"  Chunk {chunk_num}: Processed {current_idx:,}/{total_rows:,} rows")
                
                # Clean up
                del chunk, X_chunk, y_chunk
                gc.collect()
            
            print(f"✅ File {i} complete: {file_rows:,} rows processed")
        
        print(f"\n✅ Total rows loaded: {current_idx:,}")
        
        # Trim to actual size
        X_final = X_memmap[:current_idx]
        y_final = y_memmap[:current_idx]
        
        # Flush to disk
        X_final.flush()
        y_final.flush()
        
        return X_final, y_final, current_idx
    
    def fit_scaler_on_memmap(self, X_memmap, total_samples):
        """Fit scaler incrementally on memory-mapped data"""
        print("\n✅ Fitting StandardScaler incrementally on ALL data...")
        
        self.scaler = StandardScaler()
        
        # Partial fit in larger chunks for speed
        chunk_size = 200000  # 2x larger chunks
        for i in range(0, total_samples, chunk_size):
            end_idx = min(i + chunk_size, total_samples)
            chunk = X_memmap[i:end_idx]
            self.scaler.partial_fit(chunk)
            
            if i % 500000 == 0:
                print(f"  Fitted {i:,}/{total_samples:,} samples...")
        
        print("✅ Scaler fitted on full dataset")
        return self.scaler
    
    def transform_and_save(self, X_memmap, total_samples):
        """Transform data and save to new memmap"""
        print("\n✅ Scaling ALL data...")
        
        X_scaled = np.memmap('X_scaled.dat', dtype='float32', mode='w+',
                            shape=(total_samples, self.num_features))
        
        chunk_size = 200000  # 2x larger chunks
        for i in range(0, total_samples, chunk_size):
            end_idx = min(i + chunk_size, total_samples)
            chunk = X_memmap[i:end_idx]
            X_scaled[i:end_idx] = self.scaler.transform(chunk).astype(np.float32)
            
            if i % 500000 == 0:
                print(f"  Scaled {i:,}/{total_samples:,} samples...")
        
        X_scaled.flush()
        print("✅ All data scaled and saved")
        return X_scaled

In [ ]:
data_path = '/kaggle/input/datasets/saahilkapoor89/cse-cic-2018-dataset-aws-processed/CIC-IDS-2018-Dataset/CIC-IDS-2018-Dataset'
loader = FullDatasetLoader(data_path, chunk_size=700000)
X_memmap, y_memmap, total_samples = loader.load_full_dataset()

In [ ]:

print(f"\n{'='*80}")
print(f"DATASET LOADED")
print(f"{'='*80}")
print(f"✅ Total samples: {total_samples:,}")
print(f"✅ Total features: {loader.num_features} (ALL FEATURES)")
print(f"✅ Memory-mapped files created")
print(f"{'='*80}")

# Encode labels
print("\n✅ Encoding labels...")
y_array = np.array(y_memmap[:total_samples])
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_array)
num_classes = len(label_encoder.classes_)

print(f"\n✅ Classes: {num_classes}")
for idx, label in enumerate(label_encoder.classes_):
    count = np.sum(y_encoded == idx)
    print(f"   {idx}: {label:<30s} ({count:,} samples)")

del y_array
gc.collect()

loader.fit_scaler_on_memmap(X_memmap, total_samples)

X_scaled_memmap = loader.transform_and_save(X_memmap, total_samples)

# Save scaler and encoder
print("\n✅ Saving scaler and label encoder...")
with open('scaler.pkl', 'wb') as f:
    pickle.dump(loader.scaler, f)
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

del X_memmap
gc.collect()


In [ ]:
X_memmap = X_scaled_memmap

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

class ShuffledWindowGenerator(tf.keras.utils.Sequence):
    def __init__(self, X_data, y_data, indices, seq_length, batch_size, shuffle=True):
        self.X = X_data
        self.y = y_data
        self.indices = np.array(indices) # The specific list of start points for this split
        self.seq_length = seq_length
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.on_epoch_end() # Shuffle initially if True

    def __len__(self):
        return int(np.floor(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        # 1. Get the specific starting indices for this batch
        batch_indices = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        
        # 2. Create empty arrays to hold the data
        X_batch = np.zeros((self.batch_size, self.seq_length, self.X.shape[1]))
        y_batch = np.zeros((self.batch_size,))
        
        # 3. Pull the windows dynamically from the memmap based on the indices
        for i, start_idx in enumerate(batch_indices):
            end_idx = start_idx + self.seq_length
            X_batch[i] = self.X[start_idx:end_idx]
            y_batch[i] = self.y[end_idx - 1] # Target is the label of the last step
            
        return X_batch, y_batch

    def on_epoch_end(self):
        # Shuffles the indices after every epoch (Only if shuffle=True)
        if self.shuffle:
            np.random.shuffle(self.indices)

In [ ]:
# Assuming SEQ_LENGTH = 15 and BATCH_SIZE is defined (e.g., 1024 or 2048)

'''
SEQ_LENGTH = 15
# ORIGINAL_FEATURES should be dynamically set to X_memmap.shape[1] 
ORIGINAL_FEATURES = 77 
ENCODED_FEATURES = 20
BATCH_SIZE = 128 # Lowered slightly to prevent OOM
EPOCHS = 5
total_sequences = len(X_encoded_memmap) - SEQ_LENGTH + 1

print("1. Calculating all valid window starting indices...")
all_indices = np.arange(total_sequences)

print("2. Extracting target labels for stratification (this takes a moment)...")
# To balance the splits, scikit-learn needs to know the label for every window.
# The label for a window starting at 'idx' is located at 'idx + SEQ_LENGTH - 1'
window_labels = y_encoded[SEQ_LENGTH - 1 : len(y_encoded)]

print("3. Performing Stratified Split (70% Train, 15% Val, 15% Test)...")
# First split: 70% Train, 30% Temp (which will become Val + Test)
# stratify=window_labels forces every attack to be split exactly 70/30
train_idx, temp_idx, y_train_labels, y_temp_labels = train_test_split(
    all_indices, window_labels, test_size=0.30, random_state=42, stratify=window_labels
)

# Second split: Cut the 30% Temp exactly in half to get 15% Val and 15% Test
val_idx, test_idx, _, _ = train_test_split(
    temp_idx, y_temp_labels, test_size=0.50, random_state=42, stratify=y_temp_labels
)

print(f"✅ Splits complete!")
print(f"Train windows: {len(train_idx):,}")
print(f"Val windows:   {len(val_idx):,}")
print(f"Test windows:  {len(test_idx):,}")

print("\n4. Creating the Generators...")
# Train generator MUST shuffle between epochs
train_gen = ShuffledWindowGenerator(X_encoded_memmap, y_encoded, train_idx, SEQ_LENGTH, BATCH_SIZE, shuffle=True)

# Val and Test generators MUST NOT shuffle (vital for accurate evaluation!)
val_gen = ShuffledWindowGenerator(X_encoded_memmap, y_encoded, val_idx, SEQ_LENGTH, BATCH_SIZE, shuffle=False)
test_gen = ShuffledWindowGenerator(X_encoded_memmap, y_encoded, test_idx, SEQ_LENGTH, BATCH_SIZE, shuffle=False)

print("✅ Generators ready for training!")
'''

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Dense, Bidirectional, GRU, 
                                     MultiHeadAttention, LayerNormalization, 
                                     GlobalAveragePooling1D, Dropout, GaussianNoise)
from tensorflow.keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight
import os
import gc
from tensorflow.keras.callbacks import EarlyStopping

# ---------------------------------------------------------
# GPU MEMORY PROTECTOR (Must be at the very top)
# ---------------------------------------------------------
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ Enabled memory growth for {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(e)

# ==========================================
# HYPERPARAMETERS
# ==========================================
SEQ_LENGTH = 15
# ORIGINAL_FEATURES should be dynamically set to X_memmap.shape[1] 
ORIGINAL_FEATURES = 77 
ENCODED_FEATURES = 20
BATCH_SIZE = 128 # Lowered slightly to prevent OOM
EPOCHS = 5


# Assume X_memmap (scaled) and y_memmap are already loaded here...


print("\n" + "="*60)
print("PHASE 1: DENOISING AUTOENCODER (ROBUST COMPRESSION)")
print("="*60)

input_layer = Input(shape=(ORIGINAL_FEATURES,))
noisy_input = GaussianNoise(0.1)(input_layer) 
encoded = Dense(40, activation='relu')(noisy_input)
bottleneck = Dense(ENCODED_FEATURES, activation='relu', name='bottleneck_layer')(encoded)
decoded = Dense(40, activation='relu')(bottleneck)
output_layer = Dense(ORIGINAL_FEATURES, activation='linear')(decoded)

dae = Model(inputs=input_layer, outputs=output_layer)

# Gradient clipping kept as a standard safety net
custom_adam = Adam(learning_rate=0.001, clipvalue=1.0)
dae.compile(optimizer=custom_adam, loss='mse')

print("Training Denoising Autoencoder...")
dae.fit(X_memmap, X_memmap, epochs=3, batch_size=2048, validation_split=0.1)

encoder = Model(inputs=dae.input, outputs=dae.get_layer('bottleneck_layer').output)

print("\nCompressing Full Dataset to disk...")
total_samples = len(X_memmap)
X_encoded_memmap = np.memmap('X_encoded.dat', dtype='float32', mode='w+', 
                             shape=(total_samples, ENCODED_FEATURES))

chunk_size = 500000
for i in range(0, total_samples, chunk_size):
    end_idx = min(i + chunk_size, total_samples)
    X_encoded_memmap[i:end_idx] = encoder.predict(X_memmap[i:end_idx], batch_size=2048)

X_encoded_memmap.flush()
del dae, X_memmap
gc.collect()

total_sequences = len(X_encoded_memmap) - SEQ_LENGTH + 1

print("1. Calculating all valid window starting indices...")
all_indices = np.arange(total_sequences)

print("2. Extracting target labels for stratification (this takes a moment)...")
# To balance the splits, scikit-learn needs to know the label for every window.
# The label for a window starting at 'idx' is located at 'idx + SEQ_LENGTH - 1'
window_labels = y_encoded[SEQ_LENGTH - 1 : len(y_encoded)]

print("3. Performing Stratified Split (70% Train, 15% Val, 15% Test)...")
# First split: 70% Train, 30% Temp (which will become Val + Test)
# stratify=window_labels forces every attack to be split exactly 70/30
train_idx, temp_idx, y_train_labels, y_temp_labels = train_test_split(
    all_indices, window_labels, test_size=0.30, random_state=42, stratify=window_labels
)

# Second split: Cut the 30% Temp exactly in half to get 15% Val and 15% Test
val_idx, test_idx, _, _ = train_test_split(
    temp_idx, y_temp_labels, test_size=0.50, random_state=42, stratify=y_temp_labels
)

print(f"✅ Splits complete!")
print(f"Train windows: {len(train_idx):,}")
print(f"Val windows:   {len(val_idx):,}")
print(f"Test windows:  {len(test_idx):,}")

print("\n4. Creating the Generators...")
# Train generator MUST shuffle between epochs
train_gen = ShuffledWindowGenerator(X_encoded_memmap, y_encoded, train_idx, SEQ_LENGTH, BATCH_SIZE, shuffle=True)

# Val and Test generators MUST NOT shuffle (vital for accurate evaluation!)
val_gen = ShuffledWindowGenerator(X_encoded_memmap, y_encoded, val_idx, SEQ_LENGTH, BATCH_SIZE, shuffle=False)
test_gen = ShuffledWindowGenerator(X_encoded_memmap, y_encoded, test_idx, SEQ_LENGTH, BATCH_SIZE, shuffle=False)

print("✅ Generators ready for training!")

print("\n" + "="*60)
print("PHASE 3: BiGRU + MULTI-HEAD ATTENTION ARCHITECTURE")
print("="*60)

num_classes = len(np.unique(y_memmap))
inputs = Input(shape=(SEQ_LENGTH, ENCODED_FEATURES))

x = Bidirectional(GRU(64, return_sequences=True))(inputs)
x = Dropout(0.2)(x)
x = Bidirectional(GRU(64, return_sequences=True))(x)
x = Dropout(0.2)(x)

attention_out = MultiHeadAttention(num_heads=4, key_dim=64)(query=x, value=x, key=x)
x = LayerNormalization()(x + attention_out)

x = GlobalAveragePooling1D()(x)
outputs = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()




In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

print("Calculating class weights on the shuffled training data...")

# 1. Use y_train_labels directly from your train_test_split step
classes = np.unique(y_train_labels)

# 2. Compute the weights
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_labels)

# 3. Create the dictionary for Keras
class_weights_dict = dict(zip(classes, weights))

print("✅ Class weights calculated successfully!")

# Optional: Print them out to verify it worked (Heavy weights = Rare attacks)
for cls, weight in class_weights_dict.items():
    try:
        class_name = label_encoder.inverse_transform([int(cls)])[0]
    except:
        class_name = f"Class {cls}"
    print(f"{class_name}: Weight {weight:.4f}")

In [ ]:
print("\n" + "="*60)
print("PHASE 4: HANDLING CLASS IMBALANCE & TRAINING")
print("="*60)

early_stopper = EarlyStopping(
    monitor='val_loss',         # Watch the validation loss
    patience=2,                 # If it doesn't improve for 3 epochs in a row, stop
    min_delta=0.001,            # It must improve by at least this much to count
    restore_best_weights=True,  # Roll back to the best epoch before saving
    verbose=1
)


hhhhhhhh

In [ ]:
print("Starting training...")
history = model.fit(
    train_gen, 
    validation_data=val_gen, 
    epochs=EPOCHS, 
    class_weight=class_weights_dict,
    callbacks=[early_stopper]
    # multiprocessing left off to prevent memmap read collisions
)

model.save('bigru_attention_ids_model.h5')
print("✅ Training complete and model saved.")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

print("\n" + "="*60)
print("PHASE 5: EVALUATION ON TEST SET")
print("="*60)

print("1. Generating predictions on the unseen test set...")
# Predict returns the raw softmax probabilities for each class
y_pred_probs = model.predict(test_gen)

# Convert probabilities to predicted integer class labels (0, 1, 2...)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

print("2. Extracting exact true labels from the test generator...")
y_true_classes = []
# Iterate through the generator to perfectly align with the batches predicted above
for i in range(len(test_gen)):
    _, y_batch = test_gen[i]
    y_true_classes.extend(y_batch)

y_true_classes = np.array(y_true_classes)

print("\n" + "="*60)
print("FINAL CLASSIFICATION REPORT")
print("="*60)

# We use your label_encoder from earlier to map the integers back to strings (Benign, DoS, etc.)
# digits=4 gives us a highly precise readout for imbalanced classes
report = classification_report(
    y_true_classes, 
    y_pred_classes, 
    target_names=label_encoder.classes_, 
    digits=4
)

print(report)

# Optional: Print a quick confusion matrix summary
print("\nConfusion Matrix:")
cm = confusion_matrix(y_true_classes, y_pred_classes)
print(cm)